# 03 - Limpieza e ingeniería de características

En este notebook se realizará la limpieza y transformación del dataset integrado generado en la etapa anterior.

El objetivo es preparar un dataset limpio y documentado mediante:

- revisión de calidad,
- normalización de columnas y textos,
- tratamiento de valores nulos,
- revisión de duplicados,
- conversión de fechas,
- validación de valores,
- creación de cinco nuevas características.

El resultado final será un dataset preparado para análisis posteriores.

## 1. Importación de librerías

Se cargan las librerías necesarias para manipulación de datos, transformación de variables y manejo de rutas del proyecto.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Entorno listo correctamente")

pandas: 3.1.0
numpy: 2.5.4
Entorno listo correctamente


## 2. Configuración de rutas

Se definen las rutas principales del proyecto para acceder al dataset integrado y almacenar los resultados de esta etapa.

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datos"
DATA_PROCESSED = DATA_DIR / "procesados"
DOCUMENTS_DIR = PROJECT_ROOT / "documentos"

ruta_entrada = DATA_PROCESSED / "edtech_integrado.csv"
ruta_salida = DATA_PROCESSED / "edtech_dataset_limpio.csv"

print("Dataset de entrada:", ruta_entrada)
print("Dataset limpio:", ruta_salida)
print("Documentos:", DOCUMENTS_DIR)

Dataset de entrada: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_integrado.csv
Dataset limpio: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_dataset_limpio.csv
Documentos: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\documentos


In [3]:
print(
    "Dataset encontrado correctamente"
    if ruta_entrada.exists()
    else "Dataset NO encontrado"
)

Dataset encontrado correctamente


## 3. Carga del dataset integrado

Se carga el dataset generado en la etapa de integración de fuentes heterogéneas.

Este archivo será utilizado como punto de partida para aplicar las transformaciones de limpieza e ingeniería de características.

In [4]:
df = pd.read_csv(ruta_entrada)

print("Dataset cargado correctamente")
print("Dimensiones:", df.shape)

display(df.head())

Dataset cargado correctamente
Dimensiones: (200000, 31)


,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,course_title,subject_area,difficulty_level,duration_weeks,instructor_rating,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,Economics 384,Computer Science,Advanced,4,4.4,Video,1,1,0,1
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,Music 579,English,Beginner,4,4.7,Mixed,1,1,1,1
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,Language Learning 231,History,Beginner,7,3.9,Text-based,1,1,0,0
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,Biology 496,Art,Intermediate,5,3.8,Mixed,0,1,0,1
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,Psychology 251,Computer Science,Advanced,4,4.7,Video,1,1,1,1


## 4. Copia de trabajo

Se crea una copia del dataset original para aplicar las transformaciones sin modificar directamente los datos cargados.

In [5]:
df_original = df.copy()

print("Copia de seguridad creada")
print("Dataset original:", df_original.shape)
print("Dataset de trabajo:", df.shape)

Copia de seguridad creada
Dataset original: (200000, 31)
Dataset de trabajo: (200000, 31)


## 5. Diagnóstico inicial de calidad

En esta sección se realiza una revisión general del dataset antes de aplicar transformaciones.

Se analizarán:

- dimensiones,
- tipos de datos,
- valores nulos,
- registros duplicados,
- y valores únicos en variables categóricas.

### 5.1 Dimensiones del dataset

In [6]:
print("Filas y columnas:", df.shape)

Filas y columnas: (200000, 31)


### 5.2 Tipos de datos

Se revisan los tipos de datos actuales para identificar columnas que requieran conversión.

In [7]:
display(
    df.dtypes.to_frame("tipo_dato")
)

,tipo_dato
interaction_id,str
student_id,str
course_id,str
timestamp,str
activity_type,str
duration_seconds,int64
completion_rate,float64
score,float64
attempts,int64
device_used,str


### 5.3 Valores nulos

Se identifican las columnas que contienen valores faltantes y su porcentaje respecto al total de registros.

In [8]:
nulos = pd.DataFrame({
    "valores_nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
})

nulos = nulos[
    nulos["valores_nulos"] > 0
].sort_values(
    by="porcentaje_nulos",
    ascending=False
)

display(nulos)

,valores_nulos,porcentaje_nulos
score,133419,66.71


### 5.4 Registros duplicados

Se revisa si existen filas completamente duplicadas dentro del dataset.

In [9]:
duplicados = df.duplicated().sum()

print("Registros duplicados:", duplicados)

Registros duplicados: 0


### 5.5 Variables categóricas

Se revisan los valores únicos de las principales variables categóricas para identificar posibles diferencias de escritura, espacios o formatos inconsistentes.

In [10]:
columnas_categoricas = df.select_dtypes(
    include="object"
).columns

for columna in columnas_categoricas:
    print(f"\n=== {columna} ===")

    valores = df[columna].dropna().unique()

    print("Cantidad de valores únicos:", len(valores))
    print(valores[:20])


=== interaction_id ===
Cantidad de valores únicos: 200000
<StringArray>
['interaction_000000', 'interaction_000001', 'interaction_000002',
 'interaction_000003', 'interaction_000004', 'interaction_000005',
 'interaction_000006', 'interaction_000007', 'interaction_000008',
 'interaction_000009', 'interaction_000010', 'interaction_000011',
 'interaction_000012', 'interaction_000013', 'interaction_000014',
 'interaction_000015', 'interaction_000016', 'interaction_000017',
 'interaction_000018', 'interaction_000019']
Length: 20, dtype: str

=== student_id ===
Cantidad de valores únicos: 8000
<StringArray>
['student_02020', 'student_07410', 'student_03086', 'student_05092',
 'student_03626', 'student_02532', 'student_01299', 'student_01750',
 'student_04319', 'student_04298', 'student_04827', 'student_00034',
 'student_04229', 'student_05997', 'student_06230', 'student_05956',
 'student_05133', 'student_03171', 'student_05881', 'student_04332']
Length: 20, dtype: str

=== course_id ===
Can

C:\Users\Rocket Consulting\AppData\Local\Temp\ipykernel_13764\2976599654.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = df.select_dtypes(


In [11]:
categorias_principales = [
    "activity_type",
    "device_used",
    "education_level",
    "study_field",
    "learning_style",
    "device_preference",
    "motivation_level",
    "subject_area",
    "difficulty_level",
    "course_format"
]

for columna in categorias_principales:

    if columna in df.columns:
        print(f"\n=== {columna} ===")
        display(
            df[columna]
            .value_counts(dropna=False)
            .to_frame("frecuencia")
        )


=== activity_type ===


,frecuencia
activity_type,
Note Taking,33754
Quiz Attempt,33411
Resource Download,33383
Discussion Post,33328
Assignment Submit,33170
Video Watch,32954



=== device_used ===


,frecuencia
device_used,
Mobile,99902
Desktop,80249
Tablet,19849



=== education_level ===


,frecuencia
education_level,
Undergraduate,79373
High School,59891
Graduate,40562
Professional,20174


... (10 outputs left out)


=== difficulty_level ===


,frecuencia
difficulty_level,
Beginner,84005
Intermediate,77458
Advanced,38537



=== course_format ===


,frecuencia
course_format,
Video,92172
Interactive,46340
Mixed,41440
Text-based,20048


### Resultado del diagnóstico inicial

El diagnóstico permite establecer el estado inicial del dataset antes de realizar cualquier transformación.

Los problemas identificados serán tratados únicamente cuando exista una regla clara de limpieza o normalización, evitando modificar información válida de forma innecesaria.

## 6. Normalización de nombres de columnas y campos de texto

En esta sección se estandarizan los nombres de las columnas y se eliminan espacios innecesarios en las variables de texto.

El objetivo es mantener una estructura consistente y reducir posibles problemas derivados de diferencias de formato.

### 6.1 Normalización de nombres de columnas

Se eliminan espacios al inicio o final, se convierten los nombres a minúsculas y se sustituyen espacios por guiones bajos.

In [12]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Columnas normalizadas:")
print(df.columns.tolist())

Columnas normalizadas:
['interaction_id', 'student_id', 'course_id', 'timestamp', 'activity_type', 'duration_seconds', 'completion_rate', 'score', 'attempts', 'device_used', 'help_sought', 'peer_interaction', 'age', 'education_level', 'study_field', 'learning_style', 'device_preference', 'timezone', 'previous_online_courses', 'has_learning_disability', 'motivation_level', 'course_title', 'subject_area', 'difficulty_level', 'duration_weeks', 'instructor_rating', 'course_format', 'has_assignments', 'has_quizzes', 'has_discussion_forum', 'certificate_offered']


### 6.2 Normalización de campos de texto

Se eliminan espacios innecesarios al inicio y al final de las variables de texto.

No se modifican mayúsculas o minúsculas de forma general para evitar alterar identificadores o nombres válidos.

In [13]:
columnas_texto = df.select_dtypes(
    include="object"
).columns

for columna in columnas_texto:
    df[columna] = df[columna].apply(
        lambda valor: valor.strip()
        if isinstance(valor, str)
        else valor
    )

print("Campos de texto normalizados correctamente.")

Campos de texto normalizados correctamente.


C:\Users\Rocket Consulting\AppData\Local\Temp\ipykernel_13764\3486499506.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(


### 6.3 Validación de campos de texto

Se verifica que no permanezcan valores de texto con espacios adicionales al inicio o al final.

In [14]:
problemas_espacios = {}

for columna in columnas_texto:

    serie = df[columna].dropna()

    cantidad = serie.apply(
        lambda valor:
            isinstance(valor, str)
            and valor != valor.strip()
    ).sum()

    if cantidad > 0:
        problemas_espacios[columna] = cantidad


if problemas_espacios:

    print("Se encontraron espacios pendientes:")

    for columna, cantidad in problemas_espacios.items():
        print(f"{columna}: {cantidad}")

else:
    print("No se detectaron espacios adicionales.")

No se detectaron espacios adicionales.


### Resultado de la normalización

Los nombres de columnas fueron estandarizados y los campos de texto fueron revisados para eliminar espacios innecesarios.

No se alteraron identificadores, categorías ni valores válidos que no requirieran corrección.

## 7. Conversión de fechas

En esta sección se convierte la variable `timestamp` al tipo de dato fecha y hora.

El objetivo es asegurar que la información temporal pueda utilizarse correctamente en análisis posteriores y en la creación de nuevas características.

### 7.1 Conversión de `timestamp`

Se convierte la columna `timestamp` al tipo `datetime`, utilizando `errors="coerce"` para identificar posibles valores inválidos.

In [15]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("Tipo de dato de timestamp:")
print(df["timestamp"].dtype)

Tipo de dato de timestamp:
datetime64[us]


### 7.2 Validación de fechas

Se verifica que todos los valores hayan podido convertirse correctamente.

In [16]:
fechas_invalidas = df["timestamp"].isnull().sum()

print("Fechas inválidas:", fechas_invalidas)
print("Fecha mínima:", df["timestamp"].min())
print("Fecha máxima:", df["timestamp"].max())

Fechas inválidas: 0
Fecha mínima: 2024-06-12 13:55:11
Fecha máxima: 2025-06-12 13:50:19


### Resultado de la conversión

La variable `timestamp` fue convertida al tipo de dato fecha y hora.

Esta transformación permitirá extraer posteriormente características temporales como año, mes y semana de interacción.

## 8. Tratamiento de valores nulos

En esta sección se revisan los valores faltantes del dataset y se aplican reglas de tratamiento según el significado de cada variable.

El objetivo es evitar imputaciones incorrectas y conservar los valores nulos cuando representen una condición válida del negocio.

### 8.1 Identificación de valores nulos

In [17]:
nulos_actuales = pd.DataFrame({
    "valores_nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
})

nulos_actuales = nulos_actuales[
    nulos_actuales["valores_nulos"] > 0
]

display(nulos_actuales)

,valores_nulos,porcentaje_nulos
score,133419,66.71


### 8.2 Tratamiento de la variable `score`

La variable `score` presenta valores nulos en actividades que no generan calificación.

Estos valores no serán sustituidos por cero, ya que eso podría interpretarse incorrectamente como bajo rendimiento académico.

Se conservarán como valores nulos cuando el tipo de actividad no sea evaluable.

In [18]:
actividades_evaluables = [
    "Assignment Submit",
    "Quiz Attempt"
]

evaluables_sin_score = df[
    df["activity_type"].isin(actividades_evaluables)
    & df["score"].isnull()
]

no_evaluables_con_score = df[
    ~df["activity_type"].isin(actividades_evaluables)
    & df["score"].notnull()
]

print(
    "Actividades evaluables sin score:",
    len(evaluables_sin_score)
)

print(
    "Actividades no evaluables con score:",
    len(no_evaluables_con_score)
)

Actividades evaluables sin score: 0
Actividades no evaluables con score: 0


### 8.3 Decisión de tratamiento

No se realizará imputación sobre `score`, ya que los valores nulos están asociados a actividades donde la calificación no aplica.

Las demás variables serán conservadas sin modificación siempre que no presenten valores faltantes que requieran tratamiento.

In [19]:
print("Valores nulos después del tratamiento:")

display(
    df.isnull()
    .sum()
    .to_frame("valores_nulos")
    .query("valores_nulos > 0")
)

Valores nulos después del tratamiento:


,valores_nulos
score,133419


### Resultado del tratamiento de nulos

Los valores faltantes fueron revisados de acuerdo con su contexto.

Los valores nulos de `score` se conservaron porque representan actividades no evaluables y no errores de calidad.

## 9. Tratamiento de duplicados

En esta sección se revisan los registros duplicados del dataset.

El objetivo es identificar filas repetidas que puedan alterar cálculos, promedios o frecuencias y eliminarlas únicamente cuando exista evidencia de que representan duplicados reales.

### 9.1 Identificación de duplicados completos

In [20]:
duplicados_antes = df.duplicated().sum()

print("Registros duplicados antes del tratamiento:", duplicados_antes)

Registros duplicados antes del tratamiento: 0


### 9.2 Eliminación de duplicados

Si existen filas completamente duplicadas, se eliminan conservando la primera aparición.

Si no existen duplicados, el dataset se conserva sin cambios.

In [21]:
if duplicados_antes > 0:
    df = df.drop_duplicates().copy()
    print("Duplicados eliminados:", duplicados_antes)
else:
    print("No se encontraron registros duplicados.")

No se encontraron registros duplicados.


In [22]:
duplicados_despues = df.duplicated().sum()

print("Registros duplicados después del tratamiento:", duplicados_despues)
print("Dimensiones actuales del dataset:", df.shape)

Registros duplicados después del tratamiento: 0
Dimensiones actuales del dataset: (200000, 31)


### Resultado del tratamiento de duplicados

Se verificó la existencia de registros completamente duplicados.

Cuando se detectaron duplicados, estos fueron eliminados conservando una sola versión del registro. En caso de no existir duplicados, no se realizaron modificaciones innecesarias sobre el dataset.

## 10. Validación y corrección de valores inválidos

En esta sección se revisan las principales variables numéricas para confirmar que sus valores se encuentren dentro de rangos válidos.

Si se detectan valores inválidos, se documentará y aplicará la corrección correspondiente. Si no existen anomalías, el dataset se conservará sin cambios.

### 10.1 Validación de rangos

In [23]:
reglas_validacion = {
    "age": (0, 100),
    "duration_seconds": (0, None),
    "completion_rate": (0, 1),
    "score": (0, 1),
    "attempts": (0, None),
    "previous_online_courses": (0, None),
    "instructor_rating": (0, 5)
}

resultados_validacion = []

for columna, (minimo, maximo) in reglas_validacion.items():

    if columna not in df.columns:
        continue

    serie = df[columna].dropna()

    invalidos = pd.Series(False, index=serie.index)

    if minimo is not None:
        invalidos |= serie < minimo

    if maximo is not None:
        invalidos |= serie > maximo

    resultados_validacion.append({
        "variable": columna,
        "valores_invalidos": int(invalidos.sum())
    })

df_validacion = pd.DataFrame(resultados_validacion)

display(df_validacion)

,variable,valores_invalidos
0,age,0
1,duration_seconds,0
2,completion_rate,0
3,score,0
4,attempts,0
5,previous_online_courses,0
6,instructor_rating,0


### 10.2 Validación específica de fechas

In [24]:
fechas_invalidas = df["timestamp"].isnull().sum()

print("Fechas inválidas:", fechas_invalidas)

Fechas inválidas: 0


### 10.3 Decisión de corrección

La validación permite determinar si existen valores que deban ser corregidos antes de continuar con la ingeniería de características.

In [25]:
total_invalidos = (
    df_validacion["valores_invalidos"].sum()
    + fechas_invalidas
)

if total_invalidos == 0:
    print(
        "No se detectaron valores inválidos. "
        "No se requieren correcciones."
    )
else:
    print(
        "Se detectaron valores inválidos que requieren revisión:",
        total_invalidos
    )

No se detectaron valores inválidos. No se requieren correcciones.


### Resultado de la validación

No se aplicarán correcciones automáticas cuando los valores se encuentren dentro de los rangos establecidos.

En caso de no detectarse valores inválidos, se documentará que el dataset cumple con las reglas básicas de validez y continuará sin modificaciones en esta etapa.

## 11. Ingeniería de características

En esta sección se crean cinco nuevas variables a partir de la información existente.

El objetivo es transformar datos originales en características más útiles para el análisis académico y de comportamiento de los estudiantes.

### 11.1 Características temporales

A partir de `timestamp` se crean variables de año, mes y semana de interacción.

In [26]:
df["interaction_year"] = df["timestamp"].dt.year

df["interaction_month"] = df["timestamp"].dt.month

df["interaction_week"] = (
    df["timestamp"]
    .dt.isocalendar()
    .week
    .astype(int)
)

### 11.2 Duración de interacción en minutos

La duración original se encuentra expresada en segundos.

Se crea una nueva variable en minutos para facilitar su interpretación.

In [27]:
df["duration_minutes"] = (
    df["duration_seconds"] / 60
).round(2)

### 11.3 Promedio de rendimiento por estudiante

Se calcula el promedio de `score` para cada estudiante utilizando únicamente las actividades que cuentan con calificación.

Los valores nulos de `score` no se consideran como cero.

In [28]:
promedio_estudiante = (
    df.groupby("student_id")["score"]
    .mean()
)

df["average_student_score"] = (
    df["student_id"]
    .map(promedio_estudiante)
    .round(4)
)

### 11.4 Validación de las nuevas características

Se verifica que las cinco variables hayan sido creadas correctamente y que sus valores sean coherentes con los datos originales.

In [29]:
nuevas_caracteristicas = [
    "interaction_year",
    "interaction_month",
    "interaction_week",
    "duration_minutes",
    "average_student_score"
]

display(
    df[nuevas_caracteristicas]
    .head()
)

,interaction_year,interaction_month,interaction_week,duration_minutes,average_student_score
0,2024,10,43,6.53,0.7837
1,2024,11,46,28.05,0.8064
2,2024,6,26,20.25,0.8422
3,2024,11,47,4.57,0.7217
4,2024,6,26,13.68,0.8467


In [30]:
display(
    df[nuevas_caracteristicas]
    .dtypes
    .to_frame("tipo_dato")
)

,tipo_dato
interaction_year,int32
interaction_month,int32
interaction_week,int64
duration_minutes,float64
average_student_score,float64


In [31]:
display(
    df[nuevas_caracteristicas]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
interaction_year,200000.0,2024.444100,0.496867,2024.000,2024.0000,2024.00,2025.0000,2025.00
interaction_month,200000.0,6.534050,3.448346,1.000,4.0000,7.00,10.0000,12.00
interaction_week,200000.0,26.525630,14.997303,1.000,14.0000,27.00,40.0000,52.00
duration_minutes,200000.0,12.030975,10.981825,0.000,3.8200,9.02,17.5200,169.18
average_student_score,199962.0,0.800117,0.041922,0.615,0.7725,0.80,0.8275,0.98


In [32]:
print(
    "Estudiantes con average_student_score:",
    df.loc[
        df["average_student_score"].notna(),
        "student_id"
    ].nunique()
)

print(
    "Estudiantes sin average_student_score:",
    df.loc[
        df["average_student_score"].isna(),
        "student_id"
    ].nunique()
)

Estudiantes con average_student_score: 7998
Estudiantes sin average_student_score: 2


### Resultado de la ingeniería de características

Se crearon cinco nuevas características:

- `interaction_year`: año de la interacción.
- `interaction_month`: mes de la interacción.
- `interaction_week`: semana del año.
- `duration_minutes`: duración de la actividad expresada en minutos.
- `average_student_score`: promedio de rendimiento del estudiante en actividades evaluables.

Estas variables amplían la información disponible sin modificar los datos originales y podrán utilizarse en análisis posteriores.

## 12. Validación final del dataset limpio

En esta sección se realiza una revisión final del dataset después de aplicar las transformaciones de limpieza e ingeniería de características.

El objetivo es confirmar que:

- las nuevas variables fueron creadas correctamente,
- no se introdujeron duplicados,
- los tipos de datos son adecuados,
- los valores nulos esperados se mantienen bajo control,
- y el dataset conserva una estructura consistente.

### 12.1 Dimensiones finales del dataset

In [33]:
print("Dimensiones originales:", df_original.shape)
print("Dimensiones finales:", df.shape)

Dimensiones originales: (200000, 31)
Dimensiones finales: (200000, 36)


### 12.2 Validación de duplicados

In [34]:
duplicados_finales = df.duplicated().sum()

print("Registros duplicados finales:", duplicados_finales)

Registros duplicados finales: 0


### 12.3 Valores nulos finales

Se revisan nuevamente los valores nulos después de las transformaciones realizadas.

In [35]:
nulos_finales = pd.DataFrame({
    "valores_nulos": df.isnull().sum(),
    "porcentaje_nulos": (
        df.isnull().mean() * 100
    ).round(2)
})

nulos_finales = nulos_finales[
    nulos_finales["valores_nulos"] > 0
]

display(nulos_finales)

,valores_nulos,porcentaje_nulos
score,133419,66.71
average_student_score,38,0.02


### 12.4 Tipos de datos finales

In [36]:
columnas_validar = [
    "timestamp",
    "interaction_year",
    "interaction_month",
    "interaction_week",
    "duration_minutes",
    "average_student_score"
]

display(
    df[columnas_validar]
    .dtypes
    .to_frame("tipo_dato")
)

,tipo_dato
timestamp,datetime64[us]
interaction_year,int32
interaction_month,int32
interaction_week,int64
duration_minutes,float64
average_student_score,float64


### 12.5 Validación de características temporales

Se verifica que las variables derivadas de `timestamp` se encuentren dentro de rangos válidos.

In [37]:
print(
    "Años:",
    sorted(df["interaction_year"].dropna().unique())
)

print(
    "Mes mínimo:",
    df["interaction_month"].min()
)

print(
    "Mes máximo:",
    df["interaction_month"].max()
)

print(
    "Semana mínima:",
    df["interaction_week"].min()
)

print(
    "Semana máxima:",
    df["interaction_week"].max()
)

Años: [np.int32(2024), np.int32(2025)]
Mes mínimo: 1
Mes máximo: 12
Semana mínima: 1
Semana máxima: 52


### 12.6 Validación de duración en minutos

In [38]:
print(
    "Duración mínima:",
    df["duration_minutes"].min()
)

print(
    "Duración máxima:",
    df["duration_minutes"].max()
)

print(
    "Duraciones negativas:",
    (df["duration_minutes"] < 0).sum()
)

Duración mínima: 0.0
Duración máxima: 169.18
Duraciones negativas: 0


### 12.7 Validación del promedio por estudiante

Se verifica que el promedio de rendimiento permanezca dentro de la escala original de `score`.

In [39]:
print(
    "Promedio mínimo por estudiante:",
    df["average_student_score"].min()
)

print(
    "Promedio máximo por estudiante:",
    df["average_student_score"].max()
)

fuera_rango_promedio = (
    df["average_student_score"]
    .dropna()
    .between(0, 1)
    .eq(False)
    .sum()
)

print(
    "Promedios fuera de rango:",
    fuera_rango_promedio
)

Promedio mínimo por estudiante: 0.615
Promedio máximo por estudiante: 0.98
Promedios fuera de rango: 0


### 12.8 Comparación del dataset antes y después

In [40]:
comparacion = pd.DataFrame({
    "metrica": [
        "Filas",
        "Columnas",
        "Duplicados",
        "Valores nulos totales"
    ],
    "antes": [
        df_original.shape[0],
        df_original.shape[1],
        df_original.duplicated().sum(),
        df_original.isnull().sum().sum()
    ],
    "despues": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
})

display(comparacion)

,metrica,antes,despues
0,Filas,200000,200000
1,Columnas,31,36
2,Duplicados,0,0
3,Valores nulos totales,133419,133457


### Resultado de la validación final

El dataset fue validado después de aplicar las transformaciones de limpieza e ingeniería de características.

Se confirmó la consistencia de las nuevas variables, la ausencia de duplicados no deseados y la validez de los principales rangos numéricos y temporales.

Los valores nulos que permanecen corresponden a reglas previamente documentadas y no serán modificados sin justificación.

El dataset se encuentra preparado para ser almacenado como la versión limpia de esta etapa.

## 13. Generación del reporte de calidad

En esta sección se genera un reporte que resume el estado final de las variables del dataset.

El reporte incluye:

- tipo de dato,
- cantidad de valores nulos,
- porcentaje de valores nulos,
- cantidad de valores únicos,
- y observaciones sobre las principales transformaciones realizadas.

In [41]:
def obtener_transformacion(columna):

    transformaciones = {
        "timestamp": "Conversión a tipo datetime",
        "score": "Valores nulos conservados cuando la actividad no genera calificación",
        "interaction_year": "Nueva característica derivada de timestamp",
        "interaction_month": "Nueva característica derivada de timestamp",
        "interaction_week": "Nueva característica derivada de timestamp",
        "duration_minutes": "Nueva característica derivada de duration_seconds",
        "average_student_score": "Promedio de score calculado por estudiante"
    }

    return transformaciones.get(
        columna,
        "Sin transformación específica"
    )

### 13.1 Construcción del reporte

Se genera una tabla de calidad con información general de cada columna y una descripción de las transformaciones aplicadas.

In [42]:
reporte_calidad = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": [
        str(df[columna].dtype)
        for columna in df.columns
    ],
    "valores_nulos": [
        df[columna].isnull().sum()
        for columna in df.columns
    ],
    "porcentaje_nulos": [
        round(
            df[columna].isnull().mean() * 100,
            2
        )
        for columna in df.columns
    ],
    "valores_unicos": [
        df[columna].nunique()
        for columna in df.columns
    ],
    "transformacion": [
        obtener_transformacion(columna)
        for columna in df.columns
    ]
})

display(reporte_calidad)

,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos,transformacion
0,interaction_id,str,0,0.00,200000,Sin transformación específica
1,student_id,str,0,0.00,8000,Sin transformación específica
2,course_id,str,0,0.00,150,Sin transformación específica
3,timestamp,datetime64[us],0,0.00,196699,Conversión a tipo datetime
4,activity_type,str,0,0.00,6,Sin transformación específica
5,duration_seconds,int64,0,0.00,4185,Sin transformación específica
6,completion_rate,float64,0,0.00,101,Sin transformación específica
7,score,float64,133419,66.71,41,Valores nulos conservados cuando la actividad ...
8,attempts,int64,0,0.00,3,Sin transformación específica
9,device_used,str,0,0.00,3,Sin transformación específica


### 13.2 Clasificación del estado de calidad

Se asigna un estado general a cada variable considerando la presencia de valores nulos.

Los valores nulos previamente identificados como válidos no se consideran automáticamente errores.

In [43]:
def clasificar_calidad(row):

    if row["valores_nulos"] == 0:
        return "Completo"

    if row["columna"] == "score":
        return "Nulos válidos"

    if row["columna"] == "average_student_score":
        return "Revisar si aplica"

    return "Revisar"


reporte_calidad["estado_calidad"] = (
    reporte_calidad.apply(
        clasificar_calidad,
        axis=1
    )
)

display(
    reporte_calidad[
        [
            "columna",
            "tipo_dato",
            "valores_nulos",
            "porcentaje_nulos",
            "estado_calidad",
            "transformacion"
        ]
    ]
)

,columna,tipo_dato,valores_nulos,porcentaje_nulos,estado_calidad,transformacion
0,interaction_id,str,0,0.00,Completo,Sin transformación específica
1,student_id,str,0,0.00,Completo,Sin transformación específica
2,course_id,str,0,0.00,Completo,Sin transformación específica
3,timestamp,datetime64[us],0,0.00,Completo,Conversión a tipo datetime
4,activity_type,str,0,0.00,Completo,Sin transformación específica
5,duration_seconds,int64,0,0.00,Completo,Sin transformación específica
6,completion_rate,float64,0,0.00,Completo,Sin transformación específica
7,score,float64,133419,66.71,Nulos válidos,Valores nulos conservados cuando la actividad ...
8,attempts,int64,0,0.00,Completo,Sin transformación específica
9,device_used,str,0,0.00,Completo,Sin transformación específica


### 13.3 Almacenamiento del reporte

El reporte de calidad se guarda en la carpeta de documentos para mantener evidencia de las transformaciones y validaciones realizadas.

In [44]:
ruta_reporte_calidad = (
    DOCUMENTS_DIR / "reporte_calidad_guia02.csv"
)

reporte_calidad.to_csv(
    ruta_reporte_calidad,
    index=False,
    encoding="utf-8"
)

print("Reporte de calidad guardado en:")
print(ruta_reporte_calidad)

Reporte de calidad guardado en:
c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\documentos\reporte_calidad_guia02.csv


In [45]:
print(
    "Reporte creado correctamente"
    if ruta_reporte_calidad.exists()
    else "Error al crear el reporte"
)

Reporte creado correctamente


### Resultado del reporte de calidad

Se generó un reporte con el estado final de las variables del dataset, incluyendo tipos de datos, valores nulos, valores únicos y transformaciones aplicadas.

Este archivo proporciona evidencia de las decisiones tomadas durante el proceso de limpieza y facilita la trazabilidad de los cambios realizados.

## 14. Guardado del dataset limpio

En esta sección se almacena la versión final del dataset después de aplicar las transformaciones de limpieza e ingeniería de características.

El archivo resultante se guardará en la carpeta de datos procesados para conservar separados los datos originales y los datos preparados para análisis posteriores.

In [46]:
df.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8"
)

print("Dataset limpio guardado en:")
print(ruta_salida)

Dataset limpio guardado en:
c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_dataset_limpio.csv


In [47]:
print(
    "Dataset limpio creado correctamente"
    if ruta_salida.exists()
    else "Error al crear el dataset limpio"
)

Dataset limpio creado correctamente


### 14.1 Validación del archivo generado

Se vuelve a cargar el dataset limpio para comprobar que el archivo fue almacenado correctamente.

In [48]:
df_limpio_validacion = pd.read_csv(
    ruta_salida
)

print("Dimensiones del archivo guardado:")
print(df_limpio_validacion.shape)

display(
    df_limpio_validacion.head()
)

Dimensiones del archivo guardado:
(200000, 36)


,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered,interaction_year,interaction_month,interaction_week,duration_minutes,average_student_score
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,Video,1,1,0,1,2024,10,43,6.53,0.7837
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,Mixed,1,1,1,1,2024,11,46,28.05,0.8064
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,Text-based,1,1,0,0,2024,6,26,20.25,0.8422
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,Mixed,0,1,0,1,2024,11,47,4.57,0.7217
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,Video,1,1,1,1,2024,6,26,13.68,0.8467


In [49]:
nuevas_caracteristicas = [
    "interaction_year",
    "interaction_month",
    "interaction_week",
    "duration_minutes",
    "average_student_score"
]

columnas_faltantes = [
    columna
    for columna in nuevas_caracteristicas
    if columna not in df_limpio_validacion.columns
]

if len(columnas_faltantes) == 0:
    print(
        "Las cinco nuevas características fueron "
        "almacenadas correctamente."
    )
else:
    print(
        "Columnas faltantes:",
        columnas_faltantes
    )

Las cinco nuevas características fueron almacenadas correctamente.


### Resultado del almacenamiento

Se generó correctamente el archivo `edtech_dataset_limpio.csv`.

El dataset conserva las variables originales, las transformaciones aplicadas y las cinco nuevas características creadas durante la etapa de ingeniería de características.

Los archivos originales permanecen sin modificaciones para conservar la trazabilidad del proceso.

## 15. Documentación de decisiones tomadas

Durante el proceso de limpieza e ingeniería de características se documentaron las principales decisiones aplicadas al dataset.

Las transformaciones fueron realizadas únicamente cuando existía una justificación clara, evitando modificar datos válidos de forma innecesaria.

### 15.1 Decisiones de limpieza y transformación

- **Nombres de columnas:** se normalizaron eliminando espacios, utilizando minúsculas y guiones bajos.

- **Campos de texto:** se eliminaron espacios innecesarios al inicio y al final de los valores.

- **Fechas:** la columna `timestamp` fue convertida al tipo `datetime`.

- **Valores nulos:** los valores nulos de `score` fueron conservados, debido a que corresponden a actividades que no generan calificación.

- **Duplicados:** se revisaron registros completamente duplicados y solo se eliminarían en caso de existir.

- **Valores inválidos:** se validaron rangos para variables numéricas. No se realizaron correcciones cuando los datos se encontraron dentro de los límites establecidos.

- **Datos originales:** no se modificaron directamente. Todas las transformaciones fueron realizadas sobre una copia de trabajo.

- **Dataset limpio:** se generó un nuevo archivo independiente para conservar la trazabilidad entre los datos originales, integrados y procesados.

### 15.2 Nuevas características creadas

Se generaron cinco características nuevas:

1. `interaction_year`: año en que ocurrió la interacción.
2. `interaction_month`: mes en que ocurrió la interacción.
3. `interaction_week`: semana del año correspondiente a la interacción.
4. `duration_minutes`: duración de la interacción expresada en minutos.
5. `average_student_score`: promedio de rendimiento de cada estudiante calculado a partir de las actividades que generan calificación.

Estas características fueron creadas utilizando información existente y buscan facilitar análisis posteriores sin introducir datos externos o valores artificiales.

In [50]:
decisiones_transformacion = pd.DataFrame({
    "elemento": [
        "Nombres de columnas",
        "Campos de texto",
        "timestamp",
        "score",
        "Duplicados",
        "Valores inválidos",
        "interaction_year",
        "interaction_month",
        "interaction_week",
        "duration_minutes",
        "average_student_score"
    ],
    "decision": [
        "Normalización de formato",
        "Eliminación de espacios adicionales",
        "Conversión a datetime",
        "Conservar nulos válidos",
        "Eliminar únicamente duplicados completos si existen",
        "No corregir valores que cumplen los rangos definidos",
        "Crear desde timestamp",
        "Crear desde timestamp",
        "Crear desde timestamp",
        "Convertir duration_seconds a minutos",
        "Calcular promedio de score por estudiante"
    ]
})

display(decisiones_transformacion)

,elemento,decision
0,Nombres de columnas,Normalización de formato
1,Campos de texto,Eliminación de espacios adicionales
2,timestamp,Conversión a datetime
3,score,Conservar nulos válidos
4,Duplicados,Eliminar únicamente duplicados completos si ex...
5,Valores inválidos,No corregir valores que cumplen los rangos def...
6,interaction_year,Crear desde timestamp
7,interaction_month,Crear desde timestamp
8,interaction_week,Crear desde timestamp
9,duration_minutes,Convertir duration_seconds a minutos


## 16. Conclusiones

En esta etapa se realizó la limpieza y preparación del dataset integrado del proyecto EdTech Learning Analytics.

Se revisaron valores nulos, duplicados, tipos de datos, variables categóricas y rangos numéricos antes de aplicar las transformaciones necesarias.

Los valores nulos de `score` fueron conservados debido a que corresponden a actividades no evaluables y no representan errores de información.

La variable `timestamp` fue convertida correctamente a formato de fecha y hora, y los campos de texto y nombres de columnas fueron normalizados.

No se identificaron valores inválidos que requirieran correcciones adicionales.

Como parte de la ingeniería de características se generaron cinco nuevas variables relacionadas con temporalidad, duración y rendimiento académico.

Finalmente, se generó un reporte de calidad y se almacenó el dataset limpio como `edtech_dataset_limpio.csv`, manteniendo los archivos originales sin modificaciones.

El resultado de esta etapa es un conjunto de datos limpio, documentado y preparado para futuras actividades del proyecto.